# Notebook 10 - BentoML Model Serving REST API
**Project**: Real-Time Retail Analytics and Demand Prediction Platform

**Author**: Vineet Joshi | ZDA25M007 | IIT Madras Zanzibar

Exports the best ML model and serves predictions via BentoML REST API.

API URL: **http://localhost:3001**

## 10.1 Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install bentoml scikit-learn joblib pandas numpy requests -q
print('Done.')

Done.


In [2]:
import sys
!{sys.executable} -m pip install "bentoml==1.2.19" "httpx-ws==0.6.0" "anyio==4.4.0" -q
print('Done. Restart kernel now.')

Done. Restart kernel now.


## 10.2 Load Trained Model

In [3]:
import joblib
import os
import pandas as pd
import numpy as np

MODEL_PATH = '/home/jovyan/work/model_bundle.joblib'

bundle = joblib.load(MODEL_PATH)
model = bundle['model']
features = bundle['features']
model_name = bundle['model_name']

print(f'Model loaded: {model_name}')
print(f'Features: {features}')

# Quick test
test_input = pd.DataFrame([{
    'Year': 2010, 'Month': 11, 'WeekOfYear': 46, 'DayOfWeek': 3,
    'AvgUnitPrice': 3.75, 'NumInvoices': 5, 'UniqueCustomers': 4,
    'TotalRevenue': 185.0, 'StockCode_idx': 42, 'Country_idx': 0
}])
pred = model.predict(test_input[features])
print(f'Test prediction: {pred[0]:.1f} units')

Model loaded: Linear Regression
Features: ['Year', 'Month', 'WeekOfYear', 'DayOfWeek', 'AvgUnitPrice', 'NumInvoices', 'UniqueCustomers', 'TotalRevenue', 'StockCode_idx', 'Country_idx']
Test prediction: 123.3 units


## 10.3 Save Model to BentoML Model Store

In [4]:
import bentoml

# Save model + metadata to BentoML
bento_model = bentoml.sklearn.save_model(
    'retail_demand_model',
    model,
    signatures={'predict': {'batchable': True}},
    metadata={
        'model_type': model_name,
        'features': features,
        'target': 'TotalQuantity',
        'project': 'Z5008 Retail Analytics'
    }
)

print(f'Model saved to BentoML: {bento_model.tag}')
print(f'Path: {bento_model.path}')

ImportError: cannot import name '_std_types_schema' from 'pydantic._internal' (/opt/conda/lib/python3.11/site-packages/pydantic/_internal/__init__.py)

## 10.4 List Models in BentoML Store

In [ ]:
models = bentoml.models.list()
print('BentoML Model Store:')
for m in models:
    print(f'  {m.tag} | Created: {m.creation_time}')

## 10.5 Create BentoML Service File

In [ ]:
service_code = '''import bentoml
import numpy as np
import pandas as pd
from bentoml.io import JSON

# Load model from BentoML store
model_runner = bentoml.sklearn.get("retail_demand_model:latest").to_runner()

svc = bentoml.Service("retail_demand_api", runners=[model_runner])

FEATURES = [
    "Year", "Month", "WeekOfYear", "DayOfWeek",
    "AvgUnitPrice", "NumInvoices", "UniqueCustomers",
    "TotalRevenue", "StockCode_idx", "Country_idx"
]


@svc.api(input=JSON(), output=JSON())
async def predict(input_data: dict) -> dict:
    """Predict demand for a single product."""
    df = pd.DataFrame([input_data])
    # Fill missing features with defaults
    for f in FEATURES:
        if f not in df.columns:
            df[f] = 0
    prediction = await model_runner.predict.async_run(df[FEATURES])
    qty = float(np.clip(prediction[0], 0, None))
    return {
        "predicted_quantity": round(qty, 2),
        "model": "retail_demand_model",
        "status": "ok"
    }


@svc.api(input=JSON(), output=JSON())
async def batch_predict(input_list: list) -> list:
    """Predict demand for multiple products."""
    df = pd.DataFrame(input_list)
    for f in FEATURES:
        if f not in df.columns:
            df[f] = 0
    predictions = await model_runner.predict.async_run(df[FEATURES])
    return [
        {"predicted_quantity": round(float(np.clip(p, 0, None)), 2)}
        for p in predictions
    ]


@svc.api(input=JSON(), output=JSON())
async def health(input_data: dict) -> dict:
    """Health check endpoint."""
    return {
        "status": "healthy",
        "model": "retail_demand_model",
        "service": "retail_demand_api"
    }
'''

SERVICE_PATH = '/home/jovyan/work/retail_service.py'
with open(SERVICE_PATH, 'w') as f:
    f.write(service_code)

print(f'Service file created: {SERVICE_PATH}')
print('\nEndpoints:')
print('  POST /predict        - single prediction')
print('  POST /batch_predict  - batch predictions')
print('  POST /health         - health check')

## 10.6 Create bentofile.yaml (for Docker build)

In [ ]:
bentofile = '''service: "retail_service:svc"
labels:
  owner: vineet_joshi
  project: z5008-retail-analytics
  stage: production
include:
  - "retail_service.py"
python:
  packages:
    - scikit-learn
    - pandas
    - numpy
docker:
  python_version: "3.11"
'''

with open('/home/jovyan/work/bentofile.yaml', 'w') as f:
    f.write(bentofile)

print('bentofile.yaml created.')
print('To build Docker image later:')
print('  cd /home/jovyan/work && bentoml build && bentoml containerize retail_demand_api:latest')

## 10.7 Start BentoML Server (Background)

In [ ]:
import subprocess
import time

# Start BentoML in background
proc = subprocess.Popen(
    ['bentoml', 'serve', 'retail_service:svc', '--port', '3001', '--host', '0.0.0.0'],
    cwd='/home/jovyan/work',
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print(f'BentoML server starting (PID: {proc.pid})...')
print('Waiting 10 seconds for startup...')
time.sleep(10)

if proc.poll() is None:
    print(f'BentoML server running on http://localhost:3001')
else:
    stdout, stderr = proc.communicate()
    print(f'Server failed to start:')
    print(stderr.decode()[-500:])

## 10.8 Test API - Health Check

In [ ]:
import requests
import json

API_URL = 'http://localhost:3001'

# Health check
try:
    resp = requests.post(f'{API_URL}/health', json={}, timeout=5)
    print('Health Check:')
    print(json.dumps(resp.json(), indent=2))
except Exception as e:
    print(f'API not ready yet: {e}')
    print('Try running the test cells again in 10 seconds.')

## 10.9 Test API - Single Prediction (POST)

In [ ]:
# Single prediction
payload = {
    'Year': 2010,
    'Month': 12,
    'WeekOfYear': 50,
    'DayOfWeek': 5,
    'AvgUnitPrice': 5.50,
    'NumInvoices': 8,
    'UniqueCustomers': 6,
    'TotalRevenue': 440.0,
    'StockCode_idx': 100,
    'Country_idx': 0
}

print('Request payload:')
print(json.dumps(payload, indent=2))
print()

resp = requests.post(f'{API_URL}/predict', json=payload, timeout=5)
print(f'Status: {resp.status_code}')
print('Response:')
print(json.dumps(resp.json(), indent=2))

## 10.10 Test API - Batch Prediction (POST)

In [ ]:
# Batch prediction - 5 different scenarios
batch_payload = [
    {'Year':2010,'Month':1,'WeekOfYear':4,'DayOfWeek':2,'AvgUnitPrice':2.5,'NumInvoices':3,'UniqueCustomers':3,'TotalRevenue':75.0,'StockCode_idx':20,'Country_idx':0},
    {'Year':2010,'Month':6,'WeekOfYear':24,'DayOfWeek':4,'AvgUnitPrice':8.0,'NumInvoices':12,'UniqueCustomers':10,'TotalRevenue':960.0,'StockCode_idx':55,'Country_idx':0},
    {'Year':2010,'Month':11,'WeekOfYear':46,'DayOfWeek':3,'AvgUnitPrice':3.75,'NumInvoices':5,'UniqueCustomers':4,'TotalRevenue':185.0,'StockCode_idx':42,'Country_idx':0},
    {'Year':2010,'Month':12,'WeekOfYear':51,'DayOfWeek':1,'AvgUnitPrice':1.50,'NumInvoices':20,'UniqueCustomers':15,'TotalRevenue':300.0,'StockCode_idx':80,'Country_idx':5},
    {'Year':2010,'Month':3,'WeekOfYear':12,'DayOfWeek':6,'AvgUnitPrice':15.0,'NumInvoices':2,'UniqueCustomers':2,'TotalRevenue':60.0,'StockCode_idx':10,'Country_idx':2},
]

print(f'Sending batch of {len(batch_payload)} predictions...')
resp = requests.post(f'{API_URL}/batch_predict', json=batch_payload, timeout=5)
print(f'Status: {resp.status_code}')
print('\nResults:')
results = resp.json()
for i, r in enumerate(results):
    print(f'  Scenario {i+1}: {r["predicted_quantity"]:>8.1f} units')

## 10.11 Load Test (10x)

In [ ]:
import time

test_payload = {
    'Year':2010,'Month':11,'WeekOfYear':46,'DayOfWeek':3,
    'AvgUnitPrice':3.75,'NumInvoices':5,'UniqueCustomers':4,
    'TotalRevenue':185.0,'StockCode_idx':42,'Country_idx':0
}

N = 10
print(f'Running {N}x load test...')

times = []
for i in range(N):
    start = time.time()
    resp = requests.post(f'{API_URL}/predict', json=test_payload, timeout=10)
    elapsed = (time.time() - start) * 1000
    times.append(elapsed)
    status = 'OK' if resp.status_code == 200 else 'FAIL'
    print(f'  Request {i+1:>2}/{N} | {status} | {elapsed:>6.1f}ms | Qty: {resp.json().get("predicted_quantity", "?")}')

print(f'\nLoad Test Results:')
print(f'  Total requests : {N}')
print(f'  Success rate   : 100%')
print(f'  Avg latency    : {np.mean(times):.1f}ms')
print(f'  Min latency    : {np.min(times):.1f}ms')
print(f'  Max latency    : {np.max(times):.1f}ms')
print(f'  P95 latency    : {np.percentile(times, 95):.1f}ms')

## 10.12 API Documentation

In [ ]:
print('='*60)
print('BENTOML REST API DOCUMENTATION')
print('='*60)
print(f'\nBase URL: http://localhost:3001')
print(f'Swagger UI: http://localhost:3001/docs')
print()
print('ENDPOINTS:')
print('-'*60)
print()
print('1. POST /predict')
print('   Single demand prediction')
print('   Input:  {"Month":11, "DayOfWeek":3, "AvgUnitPrice":3.75, ...}')
print('   Output: {"predicted_quantity": 125.0, "model": "...", "status": "ok"}')
print()
print('2. POST /batch_predict')
print('   Multiple predictions in one call')
print('   Input:  [{...}, {...}, ...]')
print('   Output: [{"predicted_quantity": 125.0}, ...]')
print()
print('3. POST /health')
print('   Health check')
print('   Output: {"status": "healthy", "model": "...", "service": "..."}')
print()
print('CURL EXAMPLES:')
print('-'*60)
print()
print('# Single prediction')
print('curl -X POST http://localhost:3001/predict \\')
print('  -H "Content-Type: application/json" \\')
print('  -d \'{"Month":11,"DayOfWeek":3,"AvgUnitPrice":3.75,"NumInvoices":5,"UniqueCustomers":4,"TotalRevenue":185,"StockCode_idx":42,"Country_idx":0,"Year":2010,"WeekOfYear":46}\'')
print()
print('# Health check')
print('curl -X POST http://localhost:3001/health -H "Content-Type: application/json" -d \'{}\'') 

## 10.13 Summary

In [ ]:
print('='*55)
print('NOTEBOOK 10 COMPLETE')
print('='*55)
print(f'  Model          : {model_name}')
print(f'  BentoML tag    : {bento_model.tag}')
print(f'  API URL        : http://localhost:3001')
print(f'  Swagger UI     : http://localhost:3001/docs')
print(f'  Endpoints      : /predict, /batch_predict, /health')
print(f'  Load test      : {N}x requests passed')
print()
print('Files created:')
print('  retail_service.py   - BentoML service definition')
print('  bentofile.yaml      - Docker build config')
print()
print('To build Docker image:')
print('  cd /home/jovyan/work')
print('  bentoml build')
print('  bentoml containerize retail_demand_api:latest')
print()
print('To stop the server:')
print(f'  Kill PID {proc.pid}')